# SCD Type 2: **Full History**

## Initial Phase: Data Load

In [0]:
%sql
DROP TABLE IF EXISTS sandbox.ingress.customer;


In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType
from pyspark.sql.functions import max as spark_max
import random

fake_names = ["Alice Corp", "Bob Traders", "Carla Foods", "Delta Logistics", "Echo Retail",
              "Falcon Metals", "Golden Textiles", "Harbor Shipping", "Ivory Electronics", "Jupiter Motors",
              "Kappa Chemicals", "Lunar Foods", "Metro Builders", "Nova Pharma", "Orbit Airlines",
              "Prime Steel", "Quartz Mining", "River Foods", "Summit Energy", "Titan Auto"]

segments = ["AUTOMOBILE", "BUILDING", "FURNITURE", "HOUSEHOLD", "MACHINERY"]
cities = ["New York", "Chicago", "Houston", "Phoenix", "Seattle", "Miami", "Denver", "Boston"]

# Get current max c_custkey from the existing table (0 if table is empty/doesn't exist yet)
try:
    max_key_row = spark.read.table("sandbox.ingress.customer") \
        .select(spark_max("c_custkey").alias("max_key")) \
        .collect()[0]
    start_key = (max_key_row["max_key"] or 0) + 1
except Exception:
    start_key = 1

data = []
for offset in range(20):
    i = start_key + offset
    data.append((
        i,
        random.choice(fake_names),          # use random.choice, not fake_names[i-1] (i can now exceed 20)
        f"{random.randint(100,999)} Main St, {random.choice(cities)}",
        f"555-{random.randint(1000,9999)}",
        random.randint(1, 25),
        random.choice(segments)
    ))

schema = StructType([
    StructField("c_custkey", LongType(), True),
    StructField("c_name", StringType(), True),
    StructField("c_address", StringType(), True),
    StructField("c_phone", StringType(), True),
    StructField("c_nationkey", LongType(), True),
    StructField("c_mktsegment", StringType(), True),
])

df1 = spark.createDataFrame(data, schema=schema)
df1.write.mode("append").saveAsTable("sandbox.ingress.customer")

print(f"Inserted keys {start_key} through {start_key + 19}")

In [0]:
df = spark.read.table("sandbox.ingress.customer")
#display(df)
df.schema

In [0]:
# %sql
# DROP TABLE IF EXISTS sandbox.bronze.customer;
# CREATE OR REPLACE TABLE sandbox.bronze.customer (
#     id BIGINT GENERATED ALWAYS AS IDENTITY,
#     c_custkey LONG,
#     c_name STRING,
#     c_address STRING,
#     c_phone STRING,
#     c_nationkey LONG,
#     c_mktsegment STRING,
#     effective_start_date TIMESTAMP,
#     effective_end_date TIMESTAMP,
#     is_current BOOLEAN
# )
# USING DELTA

In [0]:
print(spark.read.table("sandbox.ingress.customer").count(), spark.read.table("sandbox.bronze.customer").count())

In [0]:
from pyspark.sql.functions import col, lit
from datetime import datetime,date

# Read from ingress bronze
df_ingress = spark.read.table("sandbox.ingress.customer")
df_bronze = spark.read.table("sandbox.bronze.customer")

# Create alias for both to prevent same column issue
ing_df = df_ingress.alias("ing")
brz_df = df_bronze.alias("brz")

#### Find New Records
Left anti join to find records in ing that are not in brz New Records to be inserted

In [0]:
new_df = ing_df.join(brz_df,
        col("ing.c_custkey") == col("brz.c_custkey"),
        "leftanti"
    )

new_df = new_df.withColumns({
    "effective_start_date": lit(datetime.now()),
    "effective_end_date": lit(datetime(2999, 1, 1)),
    "is_current": lit(True)
})

display(new_df)


#### Find Updated Records
We found which to be updated here we have 2 step mechanism
 1. Update old records with end date as today and is_current as false
 2. Insert new records with start date as today and is_current as true


In [0]:

insert_updated_df = ing_df.join(brz_df,
                     (col("ing.c_custkey") == col("brz.c_custkey")),
                     "inner"
    ).filter(
        (col("ing.c_name") != col("brz.c_name")) |
        (col("ing.c_address") != col("brz.c_address")) |
        (col("ing.c_phone") != col("brz.c_phone")) |
        (col("ing.c_nationkey") != col("brz.c_nationkey")) |
        (col("ing.c_mktsegment") != col("brz.c_mktsegment"))
    ).select("ing.*").withColumns({
        "effective_start_date": lit(datetime.now()),
        "effective_end_date": lit(date(2999,1,1)),
        "is_current": lit(True)    
    })

display(insert_updated_df)


In [0]:

update_old_df = ing_df.join(brz_df,
                     (col("ing.c_custkey") == col("brz.c_custkey")),
                     "inner"
    ).filter(
        (col("ing.c_name") != col("brz.c_name")) |
        (col("ing.c_address") != col("brz.c_address")) |
        (col("ing.c_phone") != col("brz.c_phone")) |
        (col("ing.c_nationkey") != col("brz.c_nationkey")) |
        (col("ing.c_mktsegment") != col("brz.c_mktsegment"))
    ).select("brz.*").withColumns({
        "effective_end_date": lit(datetime.now()),
        "is_current": lit(False)
    })

display(update_old_df)

#### Find Deleted Records

In [0]:
delete_df = brz_df.join(ing_df,
            col("brz.c_custkey") == col("ing.c_custkey"),
            "left_anti"         
            )

# To mark them as not current and set end date as today specifying it was deleted today
delete_df = delete_df.withColumns({
    "effective_end_date": lit(datetime.now()),
    "is_current": lit(False)
})

display(delete_df) 

In [0]:
from delta.tables import DeltaTable

delta_tbl = DeltaTable.forName(spark, "sandbox.bronze.customer")

delta_tbl.alias("brz").merge(
        update_old_df.alias("upd"),
        "brz.c_custkey = upd.c_custkey AND brz.is_current = true"
    ) \
    .whenMatchedUpdate(set = {
        "is_current": "false",
        "effective_end_date": "current_timestamp()"
    }) \
    .execute()

delta_tbl.alias("brz").merge(
    delete_df.alias("del"),
    "brz.c_custkey = del.c_custkey AND brz.is_current = true"
) \
    .whenMatchedUpdate(set = {
        "is_current": "false",
        "effective_end_date": "current_timestamp()"
    }) \
    .execute()


# To be inserted directly
new_df.write.mode("append").saveAsTable("sandbox.bronze.customer")
insert_updated_df.write.mode("append").saveAsTable("sandbox.bronze.customer")

In [0]:
spark.table("sandbox.bronze.customer").display()

#### Manupulating data a bit for checking updates

In [0]:
%sql
update sandbox.ingress.customer
set c_address = '123 Main Street', c_phone = '555-555-5555'
where c_custkey = 21;

delete from sandbox.ingress.customer
where c_custkey = 37;